# 原 Stage1 动态风险层：一次固定旁路干预

假说：动态风险层可能同时保护回撤并产生 SOXX 相对机会成本。固定原 Stage1 父链、0bp 与 Tiered，整层消融只做一次；不选择风险阈值。

In [1]:
import hashlib
import json
import os
import shlex
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
from qlib.workflow import R
from src.backtest.native_raw_corporate import native_raw_validate
from src.backtest.stage1_domain_heads_raw import replay_stage1_raw
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _ssh, archive_nas_recorder
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_nvda_guidance_receiver import prepare_account_sources
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.stage1_rolling126_shortfall import _analyze_account

os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
root = Path.cwd().resolve()
card = root / 'research/stage1_dynamic_risk_bypass_0bp_20261023'
plan = json.loads((card / 'plan.json').read_text())
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized']
assert plan['budget'] == {'new_market_requests': 0, 'new_model_fits': 0, 'new_native_accounts': 1, 'new_bootstrap_runs': 1, 'new_qlib_recorders': 1}
assert not (card / 'run_state.json').exists(), 'one-shot attempt already recorded'
for name, digest in plan['source_code_sha256'].items():
    assert sha(root / name) == digest, name
assert sha(root / plan['source_helper'] / 'account_source_preflight.json') == plan['source_helper_receipt_sha256']
assert sha(root / plan['provider_uri'] / 'qlib_store_manifest.json') == plan['provider_manifest_sha256']
account = prepare_account_sources(root=root, card=root / plan['source_helper'])
assert sha(root / plan['source_helper'] / 'account_source_preflight.json') == plan['source_helper_receipt_sha256']
assert len(account['frame']) == 944 and len(account['points']) == 3778
assert all(account['references'][name]['audit']['status'] == 'passed' for name in ('Stage1_A', 'SOXX'))
original_spec = IntradayRecoveryDirectAllocationSpec(root=root, **account['recovery_spec'])
ablated_spec = replace(original_spec, stop_drawdown=2.0, emergency_drawdown=2.0, trailing_loss_trigger=2.0, defensive_beta=0.0)
action_spec = Stage1OpenActionValueSpec(root=root, **account['action_spec'])
execution = {'initial_cash': plan['initial_cash_usd'], 'tariff': account['parent']['tariff']}
state = {'status': 'native_account_attempted', 'plan_sha256': sha(card / 'plan.json'), 'new_model_fits': 0, 'native_account_attempts': 1, 'bootstrap_attempts': 0}
atomic_json(state, card / 'run_state.json')
path = replay_stage1_raw(account['frame'].copy(deep=True), point_prices=account['points'], schedule=account['schedule'], plan=execution, controller=account['controller'], recovery_spec=ablated_spec, action_spec=action_spec, selection=account['selection'], slippage_bps=0)
assert len(path['replay']) == 944 and len(path['points']) == 3778
raw_beta = 3 * account['frame'].soxl_weight.to_numpy(float) + account['frame'].soxx_weight.to_numpy(float)
assert np.allclose(path['trace'].current_beta_cap.to_numpy(float), raw_beta, atol=1e-10, rtol=0)
assert path['trace'].risk_state.eq(0).all() and not path['trace'].recovery_action.any()
book = native_raw_validate(path, provider_uri=root / plan['provider_uri'], point_prices=account['points'], schedule=account['schedule'], tariff=execution['tariff'], cost=0, initial_cash=execution['initial_cash'])
assert book['audit']['status'] == 'passed'
dates = ['signal_date', 'trade_date', 'exit_date']
for name in ('Stage1_A', 'SOXX'):
    pd.testing.assert_frame_equal(book['authority'][dates].reset_index(drop=True), account['references'][name]['authority'][dates].reset_index(drop=True))
state.update(status='native_complete', native_accounts_completed=1, bootstrap_attempts=1)
atomic_json(state, card / 'run_state.json')
result = _analyze_account(account=book['authority'], reference_soxx=account['references']['SOXX']['authority'], reference_stage1=account['references']['Stage1_A']['authority'], sampled_drawdown=-100 * float(path['replay'].monitored_drawdown.max()), plan=plan)
assert len(result['annual']) == 4 and len(result['quarter']) == 16 and len(result['rolling']) == 819
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
with R.start(experiment_name=plan['experiment'], uri=str(card / 'qlib_recorder')):
    recorder = R.get_recorder()
    recorder.save_objects(artifact_path='inputs', plan=plan, source_receipt=account['receipt'])
    recorder.save_objects(artifact_path='paths', candidate_0=path)
    recorder.save_objects(artifact_path='native', candidate_0=book)
    recorder.save_objects(artifact_path='analysis', SOXX_metrics=result['SOXX_metrics'], Stage1_increment=result['Stage1_increment_metrics'], annual=result['annual'], quarter=result['quarter'], rolling_summary=result['rolling_summary'], uncertainty=result['uncertainty'], gates=result['gates'])
    recorder.save_objects(artifact_path='analysis/rolling', all_windows=result['rolling'])
    recorder.log_metrics(new_model_fits=0, native_accounts=1, SOXX_relative_wealth=result['SOXX_metrics']['relative_wealth'], historical_0bp_pass=int(result['historical_0bp_gate_pass']))
    recorder_id, experiment_id = recorder.id, str(recorder.experiment_id)
summary = {
    'experiment': plan['experiment'], 'plan_sha256': sha(card / 'plan.json'),
    'decision': 'historical_0bp_pass_needs_further_gates' if result['historical_0bp_gate_pass'] else 'not_promoted_fixed_Stage1_dynamic_risk_bypass_0bp',
    'objective_achieved': False, 'historical_0bp_gate_pass': result['historical_0bp_gate_pass'],
    'SOXX_metrics': result['SOXX_metrics'], 'Stage1_increment_metrics': result['Stage1_increment_metrics'],
    'rolling_summary': result['rolling_summary'], 'uncertainty': result['uncertainty'], 'gates': result['gates'],
    'risk_layer_ablation_audit': {'risk_state_nonzero_days': int(path['trace'].risk_state.ne(0).sum()), 'cap_equals_raw_beta_max_abs_error': float(np.max(np.abs(path['trace'].current_beta_cap.to_numpy(float) - raw_beta))), 'recovery_actions': int(path['trace'].recovery_action.sum()), 'open_execute_days': int(path['trace'].execute_open_target.sum()), 'available_mark_max_drawdown_pct': float(-100 * path['replay'].monitored_drawdown.max())},
    'new_model_fits': 0, 'new_native_accounts': 1, 'new_bootstrap_runs': 1, 'qualified_forward_days': 0,
    'recorder_id': recorder_id, 'experiment_id': experiment_id, 'recorder_uri': str(card / 'qlib_recorder'),
    'artifact_policy': {'policy': 'slim_card_recorder_first', 'workflow_config': 'workflow.yaml', 'recorder_only_artifacts': ['inputs/*', 'paths/*', 'native/*', 'analysis/rolling/*']},
    'limitations': ['Whole risk state/cap/recovery-eligibility intervention alters the frozen action model input domain; effects cannot be assigned to one risk threshold.', 'Reused 2023-2026 history is development-only.', 'Available price marks are not full-minute risk qualification.', 'Separate 2/5bp own-state accounts and independent forward evidence are absent.'],
}
atomic_json(summary, card / 'summary.json')
binding = json.loads((root / 'research/stage1_beta_composition_0bp_20261022/nas_binding.json').read_text())
binding.pop('evidence', None)
binding.update(source_id=plan['experiment'], source_plan='plan.json', as_of='2026-10-09')
atomic_json(binding, card / 'nas_binding.json')
evidence = archive_nas_recorder(card / 'nas_binding.json', workspace_root=root, card=card, recorder_id=recorder_id, experiment_id=experiment_id)
summary['nas_evidence'] = evidence
remote_sha = _ssh(binding['ssh'], 'sha256sum ' + shlex.quote(evidence['nas_path'])).split()[0]
assert remote_sha == evidence['sha256']
summary['nas_readback'] = {'sha256': remote_sha, 'bytes': evidence['size_bytes'], 'verified': True}
atomic_json(summary, card / 'summary.json')
state.update(status='completed', recorder_id=recorder_id, experiment_id=experiment_id, historical_0bp_pass=result['historical_0bp_gate_pass'])
atomic_json(state, card / 'run_state.json')
metric = result['SOXX_metrics']
complete = result['annual'].loc[result['annual'].complete_period]
positive = int(complete.relative_wealth.gt(1 + 1e-8).sum())
(card / 'conclusion.md').write_text(
    '# 原 Stage1 动态风险层旁路：固定0bp完整账户\n\n'
    f'以原 Stage1 配置/动作/恢复父链、原始股数、10万美元自身资金和独立 Tiered 费用完成 1 本0bp原生 Qlib 账户；0拟合。风险状态全程0、上限与原目标名义beta最大误差 {summary["risk_layer_ablation_audit"]["cap_equals_raw_beta_max_abs_error"]:.2e}、恢复动作0次。没有组成映射或风险参数搜索。\n\n'
    f'相对 SOXX 期末净财富 {metric["relative_wealth"]:.6f}，相对原 Stage1 财富 {result["Stage1_increment_metrics"]["relative_wealth"]:.6f}，可用点最大回撤 {abs(summary["risk_layer_ablation_audit"]["available_mark_max_drawdown_pct"]):.2f}%，CVaR5 {metric["cvar5_pct"]:.3f}%（SOXX {metric["benchmark_cvar5_pct"]:.3f}%）；完整年度 {positive}/{len(complete)} 正超额，819个连续126日窗口正超额 {result["rolling_summary"]["positive_fraction"]:.2%}，SOXX区块下界 {result["uncertainty"]["daily_lower_bound"]:.9f}。\n\n'
    f'决定 `{summary["decision"]}`。该消融把风险状态、上限和恢复资格一同改变，开盘动作模型也看到了不同输入，故只识别整层自身路径效果，不能归因于单一阈值或可交易恢复信号。历史已复用，完整分钟风险、2/5bp和独立前向仍缺；不因终值较高就称为稳定超额。全部年度/季度/126日窗口在 Recorder `{recorder_id}`，NAS SHA `{remote_sha}` 已远端回读。\n'
)
print(summary['decision'], metric['relative_wealth'], summary['risk_layer_ablation_audit']['available_mark_max_drawdown_pct'], result['rolling_summary']['positive_fraction'], result['uncertainty']['daily_lower_bound'], recorder_id)


not_promoted_fixed_Stage1_dynamic_risk_bypass_0bp 1.1194746260944337 -49.75341550638546 0.5970695970695971 -0.00038237391765237225 d6ba2b2e47754775a8ddffe57603da43


## 中文结论

固定0bp整层旁路相对SOXX终值1.119475，但可用点最大回撤49.75%、CVaR5为-5.918%、126日正超额59.71%、区块下界-0.000382374，`not_promoted`。原Stage1在2024Q3相对财富1.0789，旁路0.9397；2026Q3旁路虽优于原路径仍低于SOXX。全16季配对差异见 failure_attribution.json。该结果没有给出事前可交易的风控开关，完整分钟、2/5bp和独立前向门仍缺。

NAS归档远端哈希一致，15个Recorder工件经限定恢复逐项比对；结论卡见 conclusion.md。